# ML Lab Practical 1: Uber Fare Prediction
Predict Uber fares using Linear Regression and Random Forest Regression. Place uber.csv in the same folder as this notebook before running.

In [1]:
# 1. Import Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
# 2. Load Dataset
df = pd.read_csv("uber.csv")
# Normalize possible PDF-copy spelling errors in column names.
df.rename(columns={"dropoƯ_longitude": "dropoff_longitude", "dropoƯ_latitude": "dropoff_latitude"}, inplace=True)
print("First 5 rows:")
print(df.head())
print("\nDataset Shape:")
print(df.shape)
print("\nDataset Information:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# 3. Data Preprocessing
if "Unnamed: 0" in df.columns:
    df.drop("Unnamed: 0", axis=1, inplace=True)

df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

numeric_columns = [
    "fare_amount", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "passenger_count"
]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df.dropna(inplace=True)

In [ ]:
# 4. Remove Invalid Values
df = df[df["fare_amount"] > 0]
df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 6)]
df = df[
    df["pickup_latitude"].between(-90, 90)
    & df["dropoff_latitude"].between(-90, 90)
    & df["pickup_longitude"].between(-180, 180)
    & df["dropoff_longitude"].between(-180, 180)
].copy()

In [ ]:
# 5. Calculate Distance using Haversine Formula
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in kilometers
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    dlat = lat2 - lat1
    dlon = np.radians(lon2) - np.radians(lon1)
    a = (np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2)
    c = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    return R * c

df["distance_km"] = haversine(
    df["pickup_latitude"], df["pickup_longitude"],
    df["dropoff_latitude"], df["dropoff_longitude"]
)
df = df[df["distance_km"] > 0].copy()

In [ ]:
# 6. Identify and Remove Outliers using IQR
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    return data[(data[column] >= lower_limit) & (data[column] <= upper_limit)].copy()

print("Shape before removing outliers:", df.shape)
df = remove_outliers(df, "fare_amount")
df = remove_outliers(df, "distance_km")
print("Shape after removing outliers:", df.shape)

In [ ]:
# 7. Correlational Analysis
correlation = df[[
    "fare_amount", "passenger_count", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "distance_km"
]].corr()
print("Correlation Matrix:")
print(correlation)
plt.figure(figsize=(10, 6))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# 8. Select Features and Split Dataset
X = df[[
    "passenger_count", "pickup_longitude", "pickup_latitude",
    "dropoff_longitude", "dropoff_latitude", "distance_km"
]]
y = df["fare_amount"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [ ]:
# 9. Linear Regression Model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_prediction = linear_model.predict(X_test)

In [ ]:
# 10. Random Forest Regression Model
random_forest = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
random_forest.fit(X_train, y_train)
rf_prediction = random_forest.predict(X_test)

In [ ]:
# 11. Evaluate Models
def evaluate_model(name, actual, predicted):
    r2 = r2_score(actual, predicted)
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    print("\n" + name)
    print("---------------------------")
    print("R2 Score :", round(r2, 4))
    print("MAE      :", round(mae, 4))
    print("RMSE     :", round(rmse, 4))
    return r2, mae, rmse

linear_scores = evaluate_model("Linear Regression", y_test, linear_prediction)
rf_scores = evaluate_model("Random Forest Regression", y_test, rf_prediction)

In [ ]:
# 12. Compare Models
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regression"],
    "R2 Score": [linear_scores[0], rf_scores[0]],
    "MAE": [linear_scores[1], rf_scores[1]],
    "RMSE": [linear_scores[2], rf_scores[2]]
})
print("Model Comparison:")
display(results)

In [ ]:
# 13. Actual vs Predicted Fare
plt.figure(figsize=(8, 5))
plt.scatter(y_test, rf_prediction, alpha=0.5)
plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Actual vs Predicted Fare - Random Forest")
plt.show()